# 6.1 Attrition tables

This notebook does the following:
    - Table 1: what predicts attrition (baseline characteristics, attrited vs retained)
    - Table 2: differential attrition (does treatment predict attrition), overall and by faculty

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
import importlib
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import os
import statsmodels.formula.api as smf
import pyfixest as pf

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_balance_table import make_balance_table
from make_regression_table import make_regression_table

## (1) Prepare data for attrition analysis

In [2]:
# Load data (no pairs-only filter - attritors need to stay in the sample)
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# A lab is attrited if it has a BL survey but never completed an EL survey
labgroup_surveys = df.groupby("labgroupid")["survey"].apply(set)
attrited_labs = labgroup_surveys[~labgroup_surveys.apply(lambda s: "EL" in s)].index

df = df[df["survey"] == "BL"].copy()
df["attrited"] = df["labgroupid"].isin(attrited_labs).astype(int)

In [4]:
# Science faculty indicator
df["science_faculty"] = (df["faculty"] == "Faculty of Science (MNF)").astype(int)

# Missing survey indicator
df["missing_bl_date"] = (df["survey_date_bl"] == "").astype(int)

# Sharing equipment and space indicators
for var in ["share_equip", "share_space"]:
    df[var] = (df[f"{var}_ind"] == "Yes").astype(int)

# Waste categories
for var in ["waste_recycle", "waste_clinical", "waste_general"]:
    df[f"{var}_over3"] = df[var].isin(["3-6kg", "6-10kg", "10-12kg", ">12kg"]).astype(int)

# Specialized equipment indicators
for var in [
    "pcr",
    "ice",
    "centrifuge",
    "coffee",
    "microwave",
    "animal",
    "nonco2_incubator",
    "4c_room",
    "minus_20c_room",
    "other"
    ]:
    df[f"{var}_indicator"] = (df[f"{var}_ind"] == "Yes").astype(int)

# Any specialized equipment indicator
df["any_spec_indicator"] = df[[f"{var}_indicator" for var in [
    "pcr",
    "ice",
    "centrifuge",
    "coffee",
    "microwave",
    "animal",
    "nonco2_incubator",
    "4c_room",
    "minus_20c_room",
    "other"
]]].any(axis=1).astype(int)

In [5]:
# Variable labels (same set as the balance table)
var_labels = {
    "science_faculty": "Science Faculty",
    "no_researchers": "Number of researchers",
    "no_ft": "Number of full-time researchers",
    "annual_electricity_total": "Total",
    "annual_electricity_fc": "Fume cupboards",
    "annual_electricity_fridge": "Fridges",
    "annual_electricity_freezer": "Freezers",
    "annual_electricity_ult": "ULT freezers",
    "annual_electricity_cryostat": "Cryostats",
    "annual_electricity_microbio": "Microbiological safety cabinets",
    "annual_electricity_incubator": "CO2 incubators",
    "annual_electricity_glassware": "Glassware drying cabinets",
    "annual_electricity_bath": "Water baths",
    "annual_electricity_heater": "Block heaters",
    "annual_electricity_it": "IT equipment",
    "share_equip": "Share any equipment",
    "share_space": "Share any space",
    "waste_recycle_over3": "Recycling waste $>$ 3kg",
    "waste_clinical_over3": "Clinical waste $>$ 3kg",
    "waste_general_over3": "General waste $>$ 3kg",
    "any_spec_indicator": "Any specialized equipment",
    "pcr_indicator": "PCR machine",
    "ice_indicator": "Ice machine",
    "centrifuge_indicator": "Centrifuge",
    "coffee_indicator": "Coffee machine",
    "microwave_indicator": "Microwave",
    "animal_indicator": "Animal facility",
    "nonco2_incubator_indicator": "Non-CO2 incubator",
    "4c_room_indicator": "4\\degree C room",
    "minus_20c_room_indicator": "-20\\degree C room"
}

In [6]:
# Headers in balance table
section_headers = {
    "Baseline annual electricity consumption (kWh)": [
        "annual_electricity_total",
        "annual_electricity_fc",
        "annual_electricity_fridge",
        "annual_electricity_freezer",
        "annual_electricity_ult",
        "annual_electricity_cryostat",
        "annual_electricity_microbio",
        "annual_electricity_incubator",
        "annual_electricity_glassware",
        "annual_electricity_bath",
        "annual_electricity_heater",
        "annual_electricity_it"]}

# Annual electricity vars get 1 decimal place; everything else falls back to default_decimals
one_dec_vars = section_headers["Baseline annual electricity consumption (kWh)"]
decimals = {var: 1 for var in one_dec_vars}

# Number researchers and FT get 2 decimal places
decimals.update({var: 2 for var in ["no_researchers", "no_ft"]})

## (2) Table 1: What predicts attrition

In [7]:
# Attrited vs retained, same variables as the balance table
table = make_balance_table(
    df = df,
    treatment_var = "attrited",
    var_labels    = var_labels,
    section_headers = section_headers,
    control_val = 0,
    treatment_val = 1,
    control_label = "Retained",
    treatment_label = "Attrited",
    decimals      = decimals,
    default_decimals = 3,
    col1_width    = "9cm",
    coln_width    = "2cm",
    indent=r"\hspace{0.3cm} ",
)
out_dir = config.OUTPUT / "8_Attrition_Tables"
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "attrition_balance.tex"
_ = table_path.write_text(table)

## (3) Table 2: Differential attrition by faculty

In [8]:
# Does treatment predict attrition, overall and by faculty?
df_science = df[df["science_faculty"] == 1].copy()
df_non_science = df[df["science_faculty"] == 0].copy()

fit_all     = smf.ols("attrited ~ treated", data=df).fit(cov_type="HC1")
fit_science = smf.ols("attrited ~ treated", data=df_science).fit(cov_type="HC1")
fit_nonsci  = smf.ols("attrited ~ treated", data=df_non_science).fit(cov_type="HC1")

In [9]:
# Control-group attrition rate for each sample
control_mean_all     = df.loc[df["treated"] == 0, "attrited"].mean()
control_mean_science = df_science.loc[df_science["treated"] == 0, "attrited"].mean()
control_mean_nonsci  = df_non_science.loc[df_non_science["treated"] == 0, "attrited"].mean()

In [10]:
table = make_regression_table(
    fit_list      = [fit_all, fit_science, fit_nonsci],
    model_names   = ["(1)", "(2)", "(3)"],
    keep_vars     = ["treated"],
    var_labels    = {"treated": "Treated"},
    col_groups    = {"All": [0], "Science Faculty": [1], "Non-Science Faculty": [2]},
    baseline_mean = {0: control_mean_all, 1: control_mean_science, 2: control_mean_nonsci},
    baseline_mean_label = "Control mean",
    decimals      = 3,
    mean_decimals = 3,
    col1_width    = "5.5cm",
    coln_width    = "3cm",
)
table_path = out_dir / "attrition_differential.tex"
_ = table_path.write_text(table)